# **Merge CSV, Convert to DF, and Clean Data**

In [41]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define the directory path in Google Drive
drive_dir = '/content/drive/MyDrive/2025 Fall/Capstone/CAPSTONE WORK PROGRESS/Data/SafeGraph/orig/csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
import pandas as pd
import os

# Get a list of all CSV files in the directory
csv_files = [f for f in os.listdir(drive_dir) if f.endswith('.csv')]

# Create an empty list to store dataframes
dfs = []

# Read each CSV file and append it to the list of dataframes
for csv_file in csv_files:
    file_path = os.path.join(drive_dir, csv_file)
    try:
        df = pd.read_csv(file_path)
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {csv_file}: {e}")

# Concatenate all dataframes into a single dataframe
if dfs:
    merged_df = pd.concat(dfs, ignore_index=True)
    print("Successfully merged all CSV files.")
    display(merged_df.head())
else:
    print("No CSV files found in the specified directory.")

Successfully merged all CSV files.


,BRANDS,CATEGORY_TAGS,CITY,CLOSED_ON,DOMAINS,ENCLOSED,GEOMETRY_TYPE,INCLUDES_PARKING_LOT,ISO_COUNTRY_CODE,IS_SYNTHETIC,...,POLYGON_WKT,POSTAL_CODE,REGION,STORE_ID,STREET_ADDRESS,SUB_CATEGORY,TOP_CATEGORY,TRACKING_CLOSED_SINCE,WEBSITE,WKT_AREA_SQ_METERS
0,"[{""safegraph_brand_id"":""SG_BRAND_75c3b00240b5d...","[""Assisted Living"",""Elder Care"",""Hospice"",""Ret...",London,NaN,"[""anchor.org.uk""]",False,POLYGON,False,GB,False,...,"POLYGON ((-0.059743473978544 51.5481748504654,...",E8 1ND,Greater London,NaN,Marcon Place,Assisted Living Facilities for the Elderly,Continuing Care Retirement Communities and Ass...,2024-05-01,NaN,698.0
1,[],"[""Property Management""]",Chislehurst,NaN,[],False,POLYGON,False,GB,False,...,"POLYGON ((0.0778077734389 51.413323672, 0.0777...",BR7 6LH,Greater London,NaN,1 Bromley Lane,Residential Property Managers,Activities Related to Real Estate,NaN,NaN,339.0
2,[],[],London,NaN,"[""onthe.house""]",False,POLYGON,False,GB,False,...,POLYGON ((-0.141459898702315 51.51437048068486...,W1B 2ED,Greater London,NaN,London,Offices of Real Estate Agents and Brokers,Offices of Real Estate Agents and Brokers,NaN,https://onthe.house/,3597.0
3,[],[],London,NaN,"[""hackneytravel.co.uk""]",False,POLYGON,NaN,GB,True,...,"POLYGON ((-0.0524654722210008 51.55089, -0.052...",E5 0NS,Greater London,NaN,31 Lower Clapton Road Mishko House,Travel Agencies,Travel Arrangement and Reservation Services,NaN,NaN,1236.0
4,[],"[""General Dentistry"",""Hygiene"",""Orthodontics""]",London,NaN,"[""fastbraces.com""]",False,POLYGON,False,GB,False,...,"POLYGON ((-0.10397611914552 51.53582237062465,...",N1 0NY,Greater London,NaN,66 Upper Street,Offices of Dentists,Offices of Dentists,NaN,NaN,211.0


# **Convert to GeoDF and clip to Greater London**

In [43]:
import geopandas as gpd
from shapely import wkt

# Function to safely convert WKT to geometry
def safe_wkt_loads(wkt_string):
    try:
        return wkt.loads(wkt_string)
    except Exception:
        return None

# Convert 'POLYGON_WKT' column to geometry objects using the safe function
merged_df['geometry'] = merged_df['POLYGON_WKT'].apply(safe_wkt_loads)

# Create a GeoDataFrame
geodf = gpd.GeoDataFrame(merged_df, geometry='geometry')

# Set the Coordinate Reference System (CRS) for the GeoDataFrame to WGS 84 (EPSG:4326)
geodf.crs = "EPSG:4326"

# Reproject CRS to British National Grid (EPSG:27700)
geodf_reprojected = geodf.to_crs("EPSG:27700")

# Check the geometry types in the resulting GeoDataFrame
print("\nDistribution of geometry types in geodf_reprojected:")
print(geodf_reprojected.geometry.geom_type.value_counts())

# Display the GeoDataFrame
display(geodf_reprojected.head(2))


Distribution of geometry types in geodf_reprojected:
Polygon         299392
MultiPolygon       393
Name: count, dtype: int64


,BRANDS,CATEGORY_TAGS,CITY,CLOSED_ON,DOMAINS,ENCLOSED,GEOMETRY_TYPE,INCLUDES_PARKING_LOT,ISO_COUNTRY_CODE,IS_SYNTHETIC,...,POSTAL_CODE,REGION,STORE_ID,STREET_ADDRESS,SUB_CATEGORY,TOP_CATEGORY,TRACKING_CLOSED_SINCE,WEBSITE,WKT_AREA_SQ_METERS,geometry
0,"[{""safegraph_brand_id"":""SG_BRAND_75c3b00240b5d...","[""Assisted Living"",""Elder Care"",""Hospice"",""Ret...",London,NaN,"[""anchor.org.uk""]",False,POLYGON,False,GB,False,...,E8 1ND,Greater London,NaN,Marcon Place,Assisted Living Facilities for the Elderly,Continuing Care Retirement Communities and Ass...,2024-05-01,NaN,698.0,"POLYGON ((534631.281 185037.38, 534661.131 185..."
1,[],"[""Property Management""]",Chislehurst,NaN,[],False,POLYGON,False,GB,False,...,BR7 6LH,Greater London,NaN,1 Bromley Lane,Residential Property Managers,Activities Related to Real Estate,NaN,NaN,339.0,"POLYGON ((544593.577 170303.892, 544592.112 17..."


In [44]:
greater_london_boundary_path = '/content/drive/MyDrive/2025 Fall/Capstone/CAPSTONE WORK PROGRESS/Data/Boundaries/gla/London_GLA_Boundary.shp'
greater_london_boundary_gdf = gpd.read_file(greater_london_boundary_path)

print(greater_london_boundary_gdf.crs)
print(f"Number of features in greater_london_boundary_gdf: {len(greater_london_boundary_gdf)}")
print(f"Geometry type of the first feature in greater_london_boundary_gdf: {greater_london_boundary_gdf.geometry.iloc[0].geom_type}")

EPSG:27700
Number of features in greater_london_boundary_gdf: 1
Geometry type of the first feature in greater_london_boundary_gdf: Polygon


In [45]:
# Find the indices of geometries in geodf_reprojected that intersect greater_london_boundary_gdf
# The .union_all() combines all geometries in greater_london_boundary_gdf into a single geometry for efficiency (though not necessary in this case)
intersecting_indices = geodf_reprojected.sindex.query(greater_london_boundary_gdf.geometry.union_all(), predicate='intersects')

# Filter geodf_reprojected to keep only the intersecting geometries
geodf_greater_london = geodf_reprojected.iloc[intersecting_indices]

print(f"Number of features in the original geodf_reprojected: {len(geodf_reprojected)}")
print(f"Number of features intersecting the London boundary (original geometries): {len(geodf_greater_london)}")

# Check the geometry types in the resulting GeoDataFrame
print("\nDistribution of geometry types in geodf_greater_london:")
print(geodf_greater_london.geometry.geom_type.value_counts())

display(geodf_greater_london.head(2))

Number of features in the original geodf_reprojected: 322661
Number of features intersecting the London boundary (original geometries): 298558

Distribution of geometry types in geodf_greater_london:
Polygon         298165
MultiPolygon       393
Name: count, dtype: int64


,BRANDS,CATEGORY_TAGS,CITY,CLOSED_ON,DOMAINS,ENCLOSED,GEOMETRY_TYPE,INCLUDES_PARKING_LOT,ISO_COUNTRY_CODE,IS_SYNTHETIC,...,POSTAL_CODE,REGION,STORE_ID,STREET_ADDRESS,SUB_CATEGORY,TOP_CATEGORY,TRACKING_CLOSED_SINCE,WEBSITE,WKT_AREA_SQ_METERS,geometry
247326,[],"[""Massage""]",Gt Lon,NaN,"[""eroticlondonmassage24hours.com""]",False,POLYGON,False,GB,False,...,W1F 7NX,Greater London,NaN,Soho London,Other Personal Care Services,Personal Care Services,NaN,NaN,273.0,"MULTIPOLYGON (((623171.69 136052.985, 623170.7..."
176188,[],"[""Bar or Pub""]",London,NaN,"[""thehouse.party""]",False,POLYGON,False,GB,False,...,W1F 7NU,Greater London,NaN,61 Poland Street,Drinking Places (Alcoholic Beverages),Drinking Places (Alcoholic Beverages),NaN,NaN,273.0,"MULTIPOLYGON (((623171.69 136052.985, 623170.7..."


# **Export to CSV**

In [46]:
# Define the output directory and filename
output_dir = '/content/drive/MyDrive/2025 Fall/Capstone/CAPSTONE WORK PROGRESS/Data/SafeGraph/output'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

In [47]:
output_csv_path = os.path.join(output_dir, 'safegraph_data.csv')

# To export to CSV without a geometry warning, first convert geometry to WKT strings,
# then create a plain pandas DataFrame.
wkt_geometries = geodf_greater_london['geometry'].apply(lambda geom: geom.wkt if geom else None)

df_for_csv = geodf_greater_london.drop(columns=['geometry']).copy()
df_for_csv['geometry'] = wkt_geometries

# Export the DataFrame to a CSV file
df_for_csv.to_csv(output_csv_path, index=False)

print(f"DataFrame successfully exported to {output_csv_path}")

DataFrame successfully exported to /content/drive/MyDrive/2025 Fall/Capstone/CAPSTONE WORK PROGRESS/Data/SafeGraph/output/safegraph_data.csv
